In [1]:
import pandas as pd
import numpy as np

from sklearn.model_selection import train_test_split
from sklearn.feature_extraction import DictVectorizer
from sklearn.metrics import mean_squared_error

from sklearn.linear_model import LinearRegression
from sklearn.ensemble import RandomForestRegressor

ModuleNotFoundError: No module named 'sklearn'

In [ ]:
df = pd.read_csv("data/train.csv")
df.head()

In [ ]:
df.shape, df.columns
df["SalePrice"].describe()

In [ ]:
df.isnull().mean().sort_values(ascending=False).head(20)

In [ ]:
df["OverallQual"].value_counts().sort_index()  # quality rating 1–10 :contentReference[oaicite:1]{index=1}
df["Neighborhood"].value_counts().head()       # physical location in Ames :contentReference[oaicite:2]{index=2}
df["GrLivArea"].hist(bins=40)                  # above-ground living area :contentReference[oaicite:3]{index=3}

In [ ]:
df_full_train, df_test = train_test_split(df, test_size=0.2, random_state=1)
df_train, df_val = train_test_split(df_full_train, test_size=0.25, random_state=1)

len(df_train), len(df_val), len(df_test)


In [ ]:
y_train = df_train.SalePrice.values
y_val = df_val.SalePrice.values
y_test = df_test.SalePrice.values

for d in (df_train, df_val, df_test):
    d.drop(columns=["SalePrice"], inplace=True)

In [ ]:
numeric_features = [
    "OverallQual",   # overall material/finish quality
    "GrLivArea",     # above ground living area (sq ft)
    "GarageCars",    # garage capacity
    "GarageArea",    # garage size in sq ft
    "TotalBsmtSF",   # total basement area
    "FullBath",      # full baths above grade
    "YearBuilt",     # original construction year
    "LotArea"        # lot size in sq ft
]

categorical_features = [
    "Neighborhood",  # location in Ames
    "HouseStyle",    # style of dwelling
    "BldgType",      # type of dwelling (1Fam, Duplex, etc.)
    "Exterior1st",   # exterior covering
    "SaleCondition"  # normal, family sale, etc.
]

features = numeric_features + categorical_features


In [ ]:
# work on copies so you can still inspect original df if needed
for df_part in (df_train, df_val, df_test):
    df_part[numeric_features] = df_part[numeric_features].fillna(0)
    for col in categorical_features:
        df_part[col] = df_part[col].fillna("Missing")


In [ ]:
dv = DictVectorizer(sparse=False)

train_dicts = df_train[features].to_dict(orient="records")
X_train = dv.fit_transform(train_dicts)

val_dicts = df_val[features].to_dict(orient="records")
X_val = dv.transform(val_dicts)

model_lr = LinearRegression()
model_lr.fit(X_train, y_train)

y_pred = model_lr.predict(X_val)
rmse_lr = mean_squared_error(y_val, y_pred, squared=False)
rmse_lr


In [ ]:
rf = RandomForestRegressor(
    n_estimators=200,
    max_depth=10,
    random_state=1,
    n_jobs=-1
)

rf.fit(X_train, y_train)
y_pred = rf.predict(X_val)
rmse_rf = mean_squared_error(y_val, y_pred, squared=False)
rmse_rf

In [ ]:
# prepare full-train and test with the same preprocessing
df_full_train = df_full_train.copy()
df_test = df_test.copy()

for df_part in (df_full_train, df_test):
    df_part[numeric_features] = df_part[numeric_features].fillna(0)
    for col in categorical_features:
        df_part[col] = df_part[col].fillna("Missing")

y_full_train = df_full_train.SalePrice.values
y_test = df_test.SalePrice.values

df_full_train = df_full_train[features]
df_test = df_test[features]

dv = DictVectorizer(sparse=False)

X_full_train = dv.fit_transform(df_full_train.to_dict(orient="records"))
X_test = dv.transform(df_test.to_dict(orient="records"))

final_model = RandomForestRegressor(
    n_estimators=200,
    max_depth=10,
    random_state=1,
    n_jobs=-1
)

final_model.fit(X_full_train, y_full_train)
y_pred = final_model.predict(X_test)
rmse_final = mean_squared_error(y_test, y_pred, squared=False)
rmse_final
